# OpenAI Chat Completions API

OpenAI Chat Completions API from/to translation helpers

In [ ]:
#| default_exp openai_chat

## Imports

In [ ]:
#| export
import json
from collections import Counter
from fastcore.utils import *
from fastcore.meta import *
from fastspec.errors import api_error_from_event

from fastllm.types import *
from fastllm.streaming import *
from fastllm.streaming import mk_acollect_stream
from fastllm.openai_responses import norm_usage

In [ ]:
#| hide
import yaml,base64,httpx
from fastspec.spec import *
from fastspec.oapi import *
from fastcore.test import *
from toolslm.funccall import get_schema
from cachy.core import enable_cachy, disable_cachy
from importlib.resources import files

In [ ]:
enable_cachy(hdrs=('content-type',))

In [ ]:
specs_path = files('fastllm') / 'specs'
oai_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'openai.with-code-samples.yml').read_text())))
oai_spec

SpecParser(base_url='https://api.openai.com/v1', ops=241)

In [ ]:
oai_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})

In [ ]:
oai_cli.groups.keys()

dict_keys(['assistants', 'audio', 'batches', 'chat', 'completions', 'containers', 'conversations', 'embeddings', 'evals', 'files', 'fine_tuning', 'images', 'models', 'moderations', 'organization', 'projects', 'realtime', 'responses', 'threads', 'uploads', 'vector_stores', 'videos', 'skills', 'chatkit'])

In [ ]:
kimi_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['FIREWORKS_API_KEY']}"})
for op in kimi_cli.ops: op.base_url = 'https://api.fireworks.ai/inference/v1'

In [ ]:
qwen_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['FIREWORKS_API_KEY']}"})
for op in qwen_cli.ops: op.base_url = 'https://api.fireworks.ai/inference/v1'

In [ ]:
ds_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['DEEPSEEK_API_KEY']}"})
for op in ds_cli.ops: op.base_url = 'https://api.deepseek.com/v1'

In [ ]:
# qwen_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['QWEN_API_KEY']}"})
# for op in qwen_cli.ops: op.base_url = 'https://dashscope.aliyuncs.com/compatible-mode/v1'

## Normalize

Helpers to transform API responses into the internal representation types

### Tool Calls

Normalize OpenAI tool calls shape(s)

In [ ]:
def lite_mk_func(f):
    if isinstance(f, dict): return f
    return {'type':'function', 'function':get_schema(f, pname='parameters')}

In [ ]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [ ]:
sch = lite_mk_func(simple_add);sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

In [ ]:
#| export
def norm_tool_calls(resp, delta=False):
    "Extract Chat Completions tool calls as normalized tool calls, optionally from streaming delta events"
    out = []
    key = 'delta' if delta else 'message'
    if not   (tcs:= nested_idx(resp, 'choices', 0, key, 'tool_calls')): return out
    for tc in tcs:
        if not (fn:=tc.get("function")): continue
        extra = {k:v for k,v in tc.items() if k not in ("id","function")}
        args  = json.loads(fn.get("arguments")) if not delta else {'_delta': fn.get("arguments")}
        out.append(ToolCall(id=tc.get("id",""), name=fn.get("name",""), arguments=args, extra=extra))
    return out

User defined:

In [ ]:
resp = await oai_cli.chat.create_chat_completion(model='gpt-4o-mini', messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}], tools=[sch])
norm_tool_calls(resp)

[ToolCall(id='call_1qRVVAmYRJYz9CjbOhSoMa3y', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function'}),
 ToolCall(id='call_ArZMm1iFq2wc0svdzYDcYjEp', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function'})]

### Usage

Same as responses api

Text response:

In [ ]:
resp = await oai_cli.chat.create_chat_completion(model='gpt-4o-mini', messages=[{"role": "user", "content": "Hi!"}], tools=[sch])
norm_usage(resp)

Usage(prompt_tokens=46, completion_tokens=10, total_tokens=56, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 46, 'completion_tokens': 10, 'total_tokens': 56, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})

Tool call response:

In [ ]:
resp = await oai_cli.chat.create_chat_completion(model='gpt-4o-mini', messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}], tools=[sch])
norm_usage(resp)

Usage(prompt_tokens=66, completion_tokens=52, total_tokens=118, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 66, 'completion_tokens': 52, 'total_tokens': 118, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})

### Finish Reason

Normalize OpenAI finish reason shape(s)

In [ ]:
#| export
def norm_finish(resp, tcs=None):
    "Canonicalize finish_reason to OpenAI Chat values: stop, tool_calls, length, content_filter."
    r = nested_idx(resp, 'choices', 0, 'finish_reason')
    return FinishReason.tool_calls if r==FinishReason.stop and any(~L(tcs).attrgot('server')) else r 

### Non-stream Completion

Normalize OpenAI Responses API object into Completion.

**NB:** Per the OpenAI spec, `text`, `refusal`, and `summary_text` fields are all **required** — the `.get(k, "")` fallbacks are purely defensive against malformed responses. `OutputMessageContent` is a discriminated `oneOf` with exactly two variants: `OutputTextContent` (`output_text`) and `RefusalContent` (`refusal`) — no other content types exist in message blocks.

`summary` is an array of `SummaryTextContent` only — no discriminated union, just one type. So the `if s.get("type") == "summary_text"` check is technically redundant (it'll always be `summary_text`), and `text` is required.

Additionally, `Response.output`, `OutputMessage.content`, and `ReasoningItem.summary` are all **required** fields — so `.get()` with fallback defaults is defensive but not strictly necessary.


In [ ]:
#| export
def norm_parts(resp):
    "Normalize chat.completions response object into Completion."
    msg = nested_idx(resp, 'choices', 0, 'message') or {}
    parts = []
    if thinking := msg.get('reasoning_content'): parts.append(Part(type="thinking", text=thinking))
    if cts := msg.get('content'): parts.append(Part(type="text",text=cts,data=dict(citations=msg.get('annotations',[]))))
    if ref := msg.get('refusal'): parts.append(Part(type="refusal",text=ref))
    tcs = norm_tool_calls(resp)
    for tc in tcs: 
        tdata = {**tc.extra, 'id':tc.id, 'name':tc.name, 'arguments':tc.arguments, 'server':tc.server}
        parts.append(Part(type="tool_use", data=tdata))
    return parts

Users can define `norm_tool_calls`, `norm_parts`, `norm_finish` and `norm_usage` for a given provider, and `mk_completion` will be used to create the canonical `Completion` object based on those functions:

In [ ]:
norm_funcs = dict(norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage)
api_registry.register('openai_chat', **norm_funcs)

#### Text

In [ ]:
mn,api_name,vnd_nm = 'gpt-4o-mini', 'openai_chat', 'openai_chat'
resp = await oai_cli.chat.create_chat_completion(model=mn, messages=[{"role": "user", "content": "Say hi"}])
comp = mk_completion(resp, mn, api_name, vnd_nm); comp

<div class="prose" markdown="1">

Hello! How can I assist you today?

<details markdown='1'>

- model: `gpt-4o-mini-2024-07-18`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=9, completion_tokens=9, total_tokens=18, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 9, 'completion_tokens': 9, 'total_tokens': 18, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})`

</details>

</div>

#### Tool call

In [ ]:
resp = await oai_cli.chat.create_chat_completion(model=mn, messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}],tools=[sch])
comp = mk_completion(resp,mn,api_name,vnd_nm); comp

<div class="prose" markdown="1">



🔧 simple_add({'a': 3, 'b': 5})



🔧 simple_add({'a': 10, 'b': 20})


<details markdown='1'>

- model: `gpt-4o-mini-2024-07-18`
- finish_reason: `tool_calls`
- usage: `Usage(prompt_tokens=66, completion_tokens=52, total_tokens=118, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 66, 'completion_tokens': 52, 'total_tokens': 118, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})`

</details>

</div>

#### Refusal (mocked)

In [ ]:
comp = mk_completion({"model": "gpt-4o-mini", "choices": [{"index": 0, "message": {"role": "assistant", "content": None, "refusal": "I can't help with that."}, "finish_reason": "stop"}], "usage": {"prompt_tokens": 10, "completion_tokens": 5, "total_tokens": 15}}, mn, api_name, vnd_nm)
comp

<div class="prose" markdown="1">

I can't help with that.

<details markdown='1'>

- model: `gpt-4o-mini`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=10, completion_tokens=5, total_tokens=15, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 10, 'completion_tokens': 5, 'total_tokens': 15})`

</details>

</div>

### Streaming

Streaming module implements `PartAccum` and `acollect_stream` which are used to collate canonical `Delta` objects into a final `Completion` object similar to the non-streaming path while yielding text/thinking/citations. Each provider needs to implement `norm_sse_event` which will transform an event into a compatible `Delta` object which will be consumed by the streaming module

In [ ]:
class PrintStream:
    max_print = 10
    cnt = 0
    def print_stream(self,o,postproc=noop):
        if not isinstance(o, (Completion,Part)): 
            if o.get('thinking') and self.cnt<self.max_print: print('🧠',end='',flush=True)
            if (txt:=o.get('text')): print(txt,end='',flush=True)
            if citations:= o.get('citations'): print(postproc(citations),end='',flush=True)
            self.cnt+=1
        else: self.cnt = 0

In [ ]:
print_stream = PrintStream().print_stream

OpenAI chat api require argument collation, so we need to provide delta arguments in Delta object's `tool_call` like:

```py
ToolCall(id=tc.get("id",""), name=fn.get("name",""), arguments={'_delta': fn.get("arguments")}, extra=extra)
```

In [ ]:
#| export
def norm_sse_event(ev, **kwargs):
    "Normalize a chat completion stream event."
    # usage always arrives as a single final event with choices: []
    ev = obj2dict(ev)
    fin = nested_idx(ev, 'choices', 0, 'finish_reason')
    tcs = norm_tool_calls(ev, delta=True)
    if (dlt:=nested_idx(ev, 'choices', 0, 'delta')) is not None:
        text, thinking, refusal = dlt.get('content'), dlt.get('reasoning_content'), dlt.get('refusal')
    else: text, thinking, refusal = None,None,None
    if ev.get("error"): raise api_error_from_event(ev)
    return Delta(text=text, thinking=thinking, refusal=refusal, tool_calls=tcs, finish_reason=fin, usage=norm_usage(ev), raw=ev, **kwargs)

In [ ]:
ev = {'id': 'ea380758-a487-47fe-a746-b14dbb858cd5', 'object': 'chat.completion.chunk', 'created': 1777546979, 'model': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'choices': [{'index': 0, 'delta': {'content': '', 'reasoning_content': None}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 35788, 'completion_tokens': 57, 'total_tokens': 35845, 'prompt_tokens_details': {'cached_tokens': 24448}, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_cache_hit_tokens': 24448, 'prompt_cache_miss_tokens': 11340}}

In [ ]:
norm_sse_event(ev)

Delta(text='', thinking=None, refusal=None, tool_calls=[], citations=[], server_tool_result=None, finish_reason='stop', usage=Usage(prompt_tokens=35788, completion_tokens=57, total_tokens=35845, cached_tokens=24448, cache_creation_tokens=0, reasoning_tokens=32, raw={'prompt_tokens': 35788, 'completion_tokens': 57, 'total_tokens': 35845, 'prompt_tokens_details': {'cached_tokens': 24448}, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_cache_hit_tokens': 24448, 'prompt_cache_miss_tokens': 11340}), raw={'id': 'ea380758-a487-47fe-a746-b14dbb858cd5', 'object': 'chat.completion.chunk', 'created': 1777546979, 'model': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'choices': [{'index': 0, 'delta': {'content': '', 'reasoning_content': None}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 35788, 'completion_tokens': 57, 'total_tokens': 35845, 'prompt_tokens_details': {'cached_tokens': 24448}, 'completion_tokens_d

In [ ]:
ev = {'id': 'chatcmpl-chatcmpl-2018e36c87e44315b017d8d8e320589f', 'object': 'chat.completion.chunk', 'created': 1777548179, 'model': 'accounts/fireworks/models/qwen3p6-plus', 'choices': [], 'usage': {'prompt_tokens': 36355, 'total_tokens': 36701, 'completion_tokens': 346}}

In [ ]:
norm_sse_event(ev)

Delta(text=None, thinking=None, refusal=None, tool_calls=[], citations=[], server_tool_result=None, finish_reason=None, usage=Usage(prompt_tokens=36355, completion_tokens=346, total_tokens=36701, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 36355, 'total_tokens': 36701, 'completion_tokens': 346}), raw={'id': 'chatcmpl-chatcmpl-2018e36c87e44315b017d8d8e320589f', 'object': 'chat.completion.chunk', 'created': 1777548179, 'model': 'accounts/fireworks/models/qwen3p6-plus', 'choices': [], 'usage': {'prompt_tokens': 36355, 'total_tokens': 36701, 'completion_tokens': 346}})

Each provider also needs to implement `delta_index_fn` which returns a current idx and last idx and used by `PartAccum` like `part_accum.append(typ, idx, **tc_kwargs)`.

Here `d` is a `Delta` object, `typ` is the name of the event, it also takes last event's name and it's index to decide how to do collation

In [ ]:
#| export
def delta_index_fn(d, typ, last_typ, last_idx):
    'Returns accumulation index for current delta and updated last idx'
    if d.tool_calls: 
        tc_idx = nested_idx(d.tool_calls, 0, 'extra', 'index')
        return f"tool_{tc_idx}", last_idx
    if not (last_typ or last_idx): return 0,0
    if typ == last_typ: return last_idx, last_idx
    return last_idx + 1, last_idx + 1

Now we can create and register openai responses api's `acollect_stream`:

In [ ]:
#| export
@delegates(mk_acollect_stream, but=['index_fn', 'api_name'])
async def acollect_stream(resp, **kwargs):
    res = mk_acollect_stream(norm_and_yield(resp, norm_sse_event), index_fn=delta_index_fn, api_name='openai_chat', **kwargs)
    async for o in res: yield o

In [ ]:
norm_funcs = dict(norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage, acollect_stream=acollect_stream)
api_registry.register('openai_chat', **norm_funcs)

In [ ]:
api_registry.apis['openai_chat']

namespace(finalize_usage=<function fastcore.imports.noop(x=None, *args, **kwargs)>,
          norm_tool_calls=<function __main__.norm_tool_calls(resp, delta=False)>,
          norm_parts=<function __main__.norm_parts(resp)>,
          norm_finish=<function __main__.norm_finish(resp, tcs=None)>,
          norm_usage=<function fastllm.openai_responses.norm_usage(resp)>,
          acollect_stream=<function __main__.acollect_stream(resp, *, model=None, vendor_name=None, stop_callables=None)>)

The following tests demonstrate completions with streaming and non-streaming paths

#### Text

In [ ]:
mn,msgs = 'deepseek-v4-flash',[{"role": "user", "content": "Say hi"}, 
                               {"role": "assistant", "content": "Hello!"},
                               {"role": "user", "content": "What did I just say?"}]
resp = await ds_cli.chat.create_chat_completion(model=mn,messages=msgs)
comp = mk_completion(resp, mn, api_name, vnd_nm);comp

<div class="prose" markdown="1">

<details><summary>Thinking</summary>

We need to interpret the user's query: "What did I just say?" after I said "Hello!" as a response to their earlier "Say hi". The user is likely testing if I remember the context. I should respond acknowledging that they asked me to say "hi", and I said "Hello!" instead. Then clarify that I understood. So I'll answer that they just said "What did I just say?" but also refer back to the original "Say hi" command.

</details>

You just said, **"What did I just say?"** — which is a follow-up to your earlier instruction **"Say hi"**, to which I responded with **"Hello!"** instead of the literal word "hi". So you're pointing that out. My apologies for the slight variation! 😊

<details markdown='1'>

- model: `deepseek-v4-flash`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=18, completion_tokens=162, total_tokens=180, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=98, raw={'prompt_tokens': 18, 'completion_tokens': 162, 'total_tokens': 180, 'prompt_tokens_details': {'cached_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 98}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 18})`

</details>

</div>

In [ ]:
mn,msgs = 'accounts/fireworks/models/qwen3p6-plus',[{"role": "user", "content": "Say hi"}, 
                               {"role": "assistant", "content": "Hello!"},
                               {"role": "user", "content": "What did I just say?"}]
resp = await qwen_cli.chat.create_chat_completion(model=mn,messages=msgs)
comp = mk_completion(resp, mn, api_name, vnd_nm);comp

<div class="prose" markdown="1">

<details><summary>Thinking</summary>

Here's a thinking process:

1.  **Analyze User Input:**
   - User says: "What did I just say?"
   - I need to recall the immediately preceding message from the user.

2.  **Check Conversation History:**
   - Turn 1: User says "Say hi"
   - Turn 2: I respond "Hello!"
   - Turn 3: User says "What did I just say?"

3.  **Identify the Answer:**
   - The user's previous message was "Say hi".

4.  **Formulate Response:**
   - Keep it direct and accurate.
   - Example: "You just said: 'Say hi'" or "Your last message was 'Say hi'."

5.  **Self-Correction/Verification:**
   - Does it match the history? Yes.
   - Is it clear and concise? Yes.
   - No extra fluff needed.

   Final response: "You just said, 'Say hi.'"✅


</details>



You just said: **"Say hi"**

<details markdown='1'>

- model: `accounts/fireworks/deployments/qwen3p6-plus-llm-think`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=30, completion_tokens=230, total_tokens=260, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 30, 'total_tokens': 260, 'completion_tokens': 230, 'prompt_tokens_details': None})`

</details>

</div>

#### Text

In [ ]:
mn,msgs = 'gpt-4o-mini',[{"role": "user", "content": "Say hi"}]
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=msgs)
comp = mk_completion(resp, mn, api_name, vnd_nm);comp

<div class="prose" markdown="1">

Hello! How can I assist you today?

<details markdown='1'>

- model: `gpt-4o-mini-2024-07-18`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=9, completion_tokens=9, total_tokens=18, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 9, 'completion_tokens': 9, 'total_tokens': 18, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})`

</details>

</div>

In [ ]:
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=msgs,stream=True,stream_options={"include_usage": True})
async for o in acollect_stream(resp, vendor_name=vnd_nm): print_stream(o)

Hi

 there

!

 How

 can

 I

 assist

 you

 today

?

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)
comp.usage, o.usage

(Usage(prompt_tokens=9, completion_tokens=9, total_tokens=18, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 9, 'completion_tokens': 9, 'total_tokens': 18, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}),
 Usage(prompt_tokens=9, completion_tokens=10, total_tokens=19, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 9, 'completion_tokens': 10, 'total_tokens': 19, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}))

#### Thinking

OpenAI's Chat Completions API doesn't expose reasoning content — `reasoning_tokens` appear in usage but no `reasoning_content` field is returned. We use Kimi (`accounts/fireworks/models/kimi-k2p5`) via Fireworks's OpenAI-compatible API to test thinking parts in chat completions streaming.

Kimi's thinking is binary — enabled (default) or disabled. There's no `reasoning_effort` level (low/medium/high) or `budget_tokens` like OpenAI/Anthropic.

In [ ]:
mn,msgs = 'accounts/fireworks/models/kimi-k2p5',[{"role": "user", "content": 'What is 31231231 * 12312?'}]
resp = await kimi_cli.chat.create_chat_completion(model=mn,messages=msgs,reasoning_effort='low')
comp = mk_completion(resp, mn, api_name, 'fireworks_ai')

In [ ]:
resp = await kimi_cli.chat.create_chat_completion(model=mn,messages=msgs,stream=True,stream_options={"include_usage": True})
async for o in acollect_stream(resp, vendor_name='fireworks_ai'): print_stream(o)

🧠

🧠

🧠

🧠

🧠

...

384

518

916

072

)

In [ ]:
test_eq('thinking' in L(comp.message.content).attrgot('type'), True)
test_eq('thinking' in L(o.message.content).attrgot('type'), True)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)

#### Tool call

In [ ]:
def test_tcs(tcs1,tcs2):
    for tc1,tc2 in zip(tcs1,tcs2): 
        test_eq(tc1.name,tc2.name); test_eq(tc1.arguments,tc2.arguments); test_eq(tc1.server,tc2.server)

In [ ]:
mn = 'gpt-4o-mini'
msgs = [{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}]
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=msgs, tools=[sch])
comp = mk_completion(resp, mn, api_name, vnd_nm);comp

<div class="prose" markdown="1">



🔧 simple_add({'a': 3, 'b': 5})



🔧 simple_add({'a': 10, 'b': 20})


<details markdown='1'>

- model: `gpt-4o-mini-2024-07-18`
- finish_reason: `tool_calls`
- usage: `Usage(prompt_tokens=66, completion_tokens=52, total_tokens=118, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 66, 'completion_tokens': 52, 'total_tokens': 118, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})`

</details>

</div>

In [ ]:
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=msgs, tools=[sch], stream=True, stream_options={"include_usage": True})
async for o in acollect_stream(resp, vendor_name=vnd_nm): print_stream(o)

In [ ]:
test_tcs(comp.tool_calls,o.tool_calls)

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=66, completion_tokens=52, total_tokens=118, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 66, 'completion_tokens': 52, 'total_tokens': 118, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}),
 Usage(prompt_tokens=66, completion_tokens=52, total_tokens=118, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 66, 'completion_tokens': 52, 'total_tokens': 118, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}))

## Denormalize

Helpers to transform internal representation types back to API payload

### Msgs

In [ ]:
def mk_user_msg(txt): return Msg(role='user', content=[Part(type=PartType.text, text=txt)])

Helpers to translate back to provider specific inputs from our internal `Msg` representation

This is not exported -- media tool results are added later

In [ ]:
def denorm_tool_result(p:Part):
    "Convert canonical tool_result Part to OpenAI Chat tool message."
    return dict(role='tool', tool_call_id=p.data.get('id'), content=str(p.text))

This is not exported -- media inputs are added later

In [ ]:
def denorm_user(m:Msg):
    "Convert canonical user Msg to OpenAI Chat user message."
    if len(m.content) == 1 and m.content[0].type == PartType.text: return dict(role='user', content=m.content[0].text or '')
    return dict(role='user', content=[dict(type='text', text=p.text or '') for p in m.content if p.type == PartType.text])

In [ ]:
#| export
def denorm_tool_use(p:Part):
    "Convert canonical tool_use Part to OpenAI Chat tool_call dict."
    return dict(id=p.data.get('id'), type='function', function=dict(name=p.data.get('name'), arguments=json.dumps(p.data.get('arguments', '{}'))))

def denorm_assistant(m:Msg):
    "Convert canonical assistant Msg to OpenAI Chat assistant message + synthetic tool responses for server tools."
    tcs, srv_responses, texts = [], [], []
    for p in m.content:
        if p.type == PartType.tool_use:
            tcs.append(denorm_tool_use(p))
            if p.data.get('server'):
                srv_txt = f"[Server tool `{p.data['name']}` executed successfully, results are generated]"
                srv_responses.append(dict(role='tool', tool_call_id=p.data['id'], content=srv_txt))
        elif p.type == PartType.text: texts.append(p)
    msg = dict(role='assistant')
    if texts: msg['content'] = texts[0].text if len(texts)==1 else [dict(type='text', text=p.text or '') for p in texts]
    if tcs: msg['tool_calls'] = tcs
    thinking = [p for p in m.content if p.type == PartType.thinking]
    if thinking: msg['reasoning_content'] = ''.join(p.text or '' for p in thinking)
    return [msg] + srv_responses

def denorm_tool(m:Msg):
    "Convert canonical tool Msg to list of OpenAI Chat tool messages."
    return [denorm_tool_result(p) for p in m.content if p.type == PartType.tool_result]

def denorm_msgs(msgs:list[Msg]):
    "Convert list of canonical Msgs to OpenAI Chat messages."
    res = []
    for m in msgs:
        if   m.role == 'user':      res.append(denorm_user(m))
        elif m.role == 'assistant': res.extend(denorm_assistant(m))
        elif m.role == 'tool':      res.extend(denorm_tool(m))
    return res

In [ ]:
msg1 = mk_user_msg('Hi!')
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=denorm_msgs([msg1]),stream=True,stream_options={"include_usage": True})
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

<div class="prose" markdown="1">

Hello! How can I assist you today?

<details markdown='1'>

- model: `gpt-4o-mini-2024-07-18`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=9, completion_tokens=9, total_tokens=18, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 9, 'completion_tokens': 9, 'total_tokens': 18, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})`

</details>

</div>

### Tool Schemas

In [ ]:
#| export
def denorm_tool_schs(tools):
    "Passthrough — canonical format is already OpenAI Chat."
    return tools

### Tool Choice

In [ ]:
#| export
def denorm_tool_choice(v):
    "Map canonical tool_choice to OpenAI Chat format."
    _tc_modes = {'auto', 'required', 'any', 'force', 'none', 'off', 'disabled'}
    if v is None: return None
    if v in _tc_modes: return v if v in ('auto','none','required') else {'any':'required','force':'required','off':'none','disabled':'none'}[v]
    return {'type': 'function', 'function': {'name': v}}

Modes

In [ ]:
test_eq(denorm_tool_choice('required'), 'required')

Tool name

In [ ]:
test_eq(denorm_tool_choice('simple_add'), {'type': 'function', 'function': {'name': 'simple_add'}})

None

In [ ]:
test_eq(denorm_tool_choice(None), None)

### Reasoning Effort

No `disable` option maybe add later if needed. Anthropic uses the newer adaptive thinking which will error with legacy models < 4.6

In [ ]:
#| export
def denorm_reasoning(v): 
    if v is None: return None
    return v  # passthrough as reasoning_effort param

### Web Search Options

In [ ]:
#| export
def denorm_web_search(v): return v

### System

In [ ]:
#| export
def denorm_system(sp, msgs): 
    msgs.insert(0, dict(role='system', content=sys_text(part_txt(sp))))
    return msgs

In [ ]:
sp = 'You are a pirate. Always respond in pirate speak. Keep it to one sentence.'
msg1 = mk_user_msg('What are you?')

OpenAI chat prepends sp to msgs

In [ ]:
msgs = denorm_msgs([msg1])
msgs = denorm_system(sp, msgs); msgs

[{'role': 'system',
  'content': 'You are a pirate. Always respond in pirate speak. Keep it to one sentence.'},
 {'role': 'user', 'content': 'What are you?'}]

In [ ]:
resp = await oai_cli.chat.create_chat_completion(model='gpt-4o-mini',messages=msgs,stream=True,stream_options={"include_usage": True})
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

<div class="prose" markdown="1">

I be a clever parleyin' assistant, savvy?

<details markdown='1'>

- model: `gpt-4o-mini-2024-07-18`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=32, completion_tokens=12, total_tokens=44, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 32, 'completion_tokens': 12, 'total_tokens': 44, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})`

</details>

</div>

### Media Inputs

`fastllm` canonicalizes multimodal inputs into `Part(type, text, data)` where `text` carries the URL or data URL, and `data` is reserved for optional metadata. Each provider's denorm layer converts this canonical form into the provider's wire format.

In [ ]:
#| export
def denorm_user(m:Msg):
    "Convert canonical user Msg to OpenAI Chat user message."
    parts = []
    for p in m.content:
        if   p.type == PartType.text:        parts.append({"type": "text", "text": p.text or ""})
        elif p.type == PartType.input_image: parts.append(denorm_image(p))
        elif p.type == PartType.input_audio: parts.append(denorm_audio(p))
        elif p.type == PartType.input_video: raise ValueError("OpenAI Chat API does not support video input")
        elif p.type == PartType.input_file:  parts.append(denorm_file(p))
    if len(parts) == 1 and parts[0].get('type') == 'text': return dict(role='user', content=parts[0]['text'])
    return dict(role='user', content=parts)

#### Image

In [ ]:
#| export
def denorm_image(p): return {"type": "image_url", "image_url": {"url": p.text}}

Image URL

In [ ]:
img_url = "https://img.freepik.com/free-photo/mountain-range-body-water_53876-139760.jpg?semt=ais_hybrid&w=740&q=80"
msg1 = Msg(role='user', content=[Part(type=PartType.input_image, text=img_url), Part(type=PartType.text, text='What is this image?')])

In [ ]:
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=denorm_msgs([msg1]),stream=True,stream_options={"include_usage": True})
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

<div class="prose" markdown="1">

The image depicts a serene landscape featuring a lake surrounded by mountains and trees. The lake reflects the scenery, creating a peaceful and picturesque view. There's a wooden dock or platform in the foreground, adding to the natural aesthetic. The overall atmosphere suggests a tranquil outdoor setting, likely in a mountainous region.

<details markdown='1'>

- model: `gpt-4o-mini-2024-07-18`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=25513, completion_tokens=59, total_tokens=25572, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 25513, 'completion_tokens': 59, 'total_tokens': 25572, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})`

</details>

</div>

Image data

In [ ]:
img_b64 = "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAQAAAAECAIAAAAmkwkpAAAAEElEQVR4nGP4z8AARwzEcQCukw/x0F8jngAAAABJRU5ErkJggg=="
msg1 = Msg(role='user', content=[Part(type=PartType.input_image, text=img_b64), Part(type=PartType.text, text='What color is this pixel?')])

In [ ]:
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=denorm_msgs([msg1]),stream=True,stream_options={"include_usage": True})
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

<div class="prose" markdown="1">

This pixel is red.

<details markdown='1'>

- model: `gpt-4o-mini-2024-07-18`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=8513, completion_tokens=5, total_tokens=8518, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 8513, 'completion_tokens': 5, 'total_tokens': 8518, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})`

</details>

</div>

#### Audio

In [ ]:
#| export
def denorm_audio(p):
    _mime_audio_fmt = {'audio/wav':'wav', 'audio/mpeg':'mp3', 'audio/mp3':'mp3'}
    if not (b64:=data_url(p.text)): raise ValueError("OpenAI Chat audio input requires base64 data URL")
    return {"type": "input_audio", "input_audio": {"data": b64[1], "format": _mime_audio_fmt.get(b64[0], 'wav')}}

In [ ]:
wav_data = httpx.get("https://openaiassets.blob.core.windows.net/$web/API/docs/audio/alloy.wav").content
audio_b64 = f"data:audio/wav;base64,{base64.b64encode(wav_data).decode()}"

OpenAI chat requires a [specific model](https://developers.openai.com/api/docs/guides/audio?example=audio-in#add-audio-to-your-existing-application) like `gpt-4o-audio-preview`

In [ ]:
msg1 = Msg(role='user', content=[Part(type=PartType.input_audio, text=audio_b64), Part(type=PartType.text, text='What is this audio saying?')])

In [ ]:
resp = await oai_cli.chat.create_chat_completion(model='gpt-audio-1.5',messages=denorm_msgs([msg1]),stream=True,stream_options={"include_usage": True})
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

<div class="prose" markdown="1">

{"text": "The sun rises in the east and sets in the west. This simple fact has been observed by humans for thousands of years."}

<details markdown='1'>

- model: `gpt-audio-1.5`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=86, completion_tokens=30, total_tokens=116, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 86, 'completion_tokens': 30, 'total_tokens': 116, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 69, 'text_tokens': 17, 'image_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': 30}})`

</details>

</div>

#### Files

OpenAI requires a filename, Anthropic requires the media type and Gemini requires the mime type

In [ ]:
#| export
def denorm_file(p):
    if (b64:=data_url(p.text)): return {"type": "file", "file": {"file_data": p.text, "filename": f"upload.{b64[0].split('/')[-1]}"}}
    raise ValueError("OpenAI Chat file input requires base64 data URL or file_id, not URLs")

In [ ]:
pdf_url = "https://arxiv.org/pdf/1706.03762"
msg1 = Msg(role='user', content=[Part(type=PartType.input_file, text=pdf_url), Part(type=PartType.text, text='What is the title of this paper?')])

In [ ]:
try: await oai_cli.chat.create_chat_completion(model=mn,messages=denorm_msgs([msg1]),stream=True,stream_options={"include_usage": True})
except Exception as e: print(e)

OpenAI Chat file input requires base64 data URL or file_id, not URLs


In [ ]:
pdf_b64 = f"data:application/pdf;base64,{base64.b64encode(httpx.get(pdf_url).content).decode()}"
msg1 = Msg(role='user', content=[Part(type=PartType.input_file, text=pdf_b64), Part(type=PartType.text, text='What is the title of this paper')])

In [ ]:
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=denorm_msgs([msg1]),stream=True,stream_options={"include_usage": True})
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

<div class="prose" markdown="1">

The title of the paper is "Attention Is All You Need."

<details markdown='1'>

- model: `gpt-4o-mini-2024-07-18`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=11386, completion_tokens=13, total_tokens=11399, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 11386, 'completion_tokens': 13, 'total_tokens': 11399, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})`

</details>

</div>

### Media Tool Call Results

A tool call can return an `input_image` or `input_file`

In [ ]:
#| export
def denorm_tool_result(p:Part):
    "Convert canonical tool_result Part to OpenAI Chat tool message."
    if isinstance(p.text, list): raise ValueError("OpenAI Chat does not support media in tool results")
    return dict(role='tool', tool_call_id=p.data.get('id') or p.data.get('call_id', ''), content=str(p.text))

In [ ]:
msgs = [Msg(role='user', content=[Part(type=PartType.text, text="What's in this image?")]),
        Msg(role='assistant', content=[Part(type=PartType.tool_use, data={'id': '_test', 'name': 'test_img', 'arguments': {}, 'server': False})]),
        Msg(role='tool', content=[Part(type=PartType.tool_result, text=[Part(type=PartType.input_image, text=img_b64)], data={'id': '_test', 'name': 'test_img'})])]

In [ ]:
try: resp = await oai_cli.chat.create_chat_completion(model=mn,messages=denorm_msgs(msgs),stream=True,stream_options={"include_usage": True})
except Exception as e: print(e)

OpenAI Chat does not support media in tool results


## Payload

Function to create the payload:

In [ ]:
#| export
@delegates(payload_kwargs)
def mk_payload(msgs, model, **kwargs):
    payload = dict(model=model, messages=denorm_msgs(msgs))
    if sp:=kwargs.get('system'):            payload['messages'] = denorm_system(sp, payload['messages'])
    if kwargs.get('stream'):                payload.update(stream=True, stream_options={"include_usage": True})
    if mt:=kwargs.get('max_tokens'):        payload['max_tokens'] = mt
    if tools:=kwargs.get('tools'):          payload['tools'] = denorm_tool_schs(tools)
    if tchc:=kwargs.get('tool_choice'):     payload['tool_choice'] = denorm_tool_choice(tchc)
    if thk:=kwargs.get('reasoning_effort'): payload['reasoning_effort'] = denorm_reasoning(thk)
    if (wopts:=kwargs.get('web_search_options')) is not None: 
        payload['web_search_options'] = denorm_web_search(wopts)
    if (temp:=kwargs.get('temperature')) is not None: payload['temperature'] = temp
    return payload

Function to create the default headers:

In [ ]:
#| export
def get_hdrs(api_key=None):
    return {"Authorization": f"Bearer {get_api_key(api_key, 'OPENAI_API_KEY')}"}

## Cost

A function to calculate api Completion cost from raw usage data, and model metadata returned by `get_model_info`

In [ ]:
#| export
def cost(usage, m):
    raw = usage.raw
    pd,cd = raw.get('prompt_tokens_details') or {},raw.get('completion_tokens_details') or {}
    cached = pd.get('cached_tokens', 0)
    in_audio, out_audio = pd.get('audio_tokens', 0), cd.get('audio_tokens', 0)
    in_txt  = raw['prompt_tokens']     - cached - in_audio
    out_txt = raw['completion_tokens'] - out_audio
    cost  = in_txt  * m.input_cost_per_token  + out_txt * m.output_cost_per_token
    cost += cached  * m.get('cache_read_input_token_cost', 0)
    cost += in_audio  * m.get('input_cost_per_audio_token', 0)
    cost += out_audio * m.get('output_cost_per_audio_token', 0)
    return cost

## Register API

Register the final API namespace:

- `norm_tool_calls`, `norm_parts`, `norm_finish`, `norm_usage` required during the Completion object construction.
- `mk_payload` is used to construct the api request payload
- `accollect_stream` is used in streaming api call path
- `get_hdrs` is used to create the deafult request headers
- `op_path` define the non-streaming and streaming OpenAPIClient op names

In [ ]:
#| export
api_ns = dict(norm_tool_calls=norm_tool_calls,
                norm_parts=norm_parts,
                norm_finish=norm_finish,
                norm_usage=norm_usage,
                acollect_stream=acollect_stream,
                mk_payload=mk_payload,
                cost=cost,
                get_hdrs=get_hdrs, 
                op_path=('chat.create_chat_completion','chat.create_chat_completion'))
api_registry.register('openai_chat', **api_ns)

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()